In [ ]:
# Modules to install
# pip install scikit-clean pandas numpy sklearn

# Importo librerías

In [1]:
import numpy as np
import pandas as pd
from pathlib import Path
from itertools import product
import warnings
import pickle

from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier

# My libraries
from filters import *
from noisers.classes import *
from noisers.funcs import *
from testFuncs import *
from noise_cv_eval import run_5cv_experiment, run_5cv_grid, run_5cv_baseline
from cleaners import CNCNOS


rs = 33

/home/juamp/Documents/Master/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
dataset_root = Path("dataset")
from functools import lru_cache

@lru_cache
def n_instances(ds):
    return load_dataset_df(
        dataset=ds,
        noise_type="data_base",
        split="cc",
        encoding=None,
        root=dataset_root,
    ).shape[0]

n_desired_datasets=30

keel_datasets = sorted(
    [p.name for p in (dataset_root / "data_base").iterdir() if p.is_dir()],
    key=lambda ds: (n_instances(ds), ds),
)[:n_desired_datasets]
print(f"Available datasets: {keel_datasets}\nNumber={len(keel_datasets)}")


Available datasets: ['zoo', 'hayes-roth', 'lymphography', 'iris', 'autos', 'wine', 'sonar', 'glass', 'newthyroid', 'heart', 'cleveland', 'splice', 'ecoli', 'ionosphere', 'dermatology', 'monk-2', 'led7digit', 'wdbc', 'balance', 'pima', 'vehicle', 'vowel', 'german', 'flare', 'nursery', 'contraceptive', 'yeast', 'car', 'shuttle', 'segment']
Number=30


### Display available filters

In [3]:
print_available_filters()


['AllKNN',
 'TomekLinks',
 'ENNFilter',
 'ENNProb',
 'ENNTh',
 'MultiEditFilter',
 'NCNEdit',
 'ClassificationFilter',
 'CVCFFilter',
 'EnsembleFiltering',
 'INFFCFilter',
 'IterativePartitioningFilter',
 'TABPFNClassificationFilter']

# Técnicas frente a las que comparar

### Filtros basados en distancia

In [ ]:
from sklearn.linear_model import LogisticRegression


# Rejilla global de hiperparámetros
ks = [3, 5, 9]   # Número de vecinos
thresholds = [0.5, 0.7]  # Cota probabilística
n_blocks = [5, 7] 
max_iters = [3]

def build_distance_grid(ks, thresholds, n_blocks):
    '''
    Returns a list of filters, eaach with a different hyperparameter config
    '''
    distance_grid = []
    # ENN base
    for k in ks:
        distance_grid.append({
            "name": f"ENN_k{k}",
            "filter": ENNFilter(n_neighbors=k, mode="enn", n_jobs=-1),
        })

    # ENNProb y ENNProb+Th
    for k in ks:
        distance_grid.append({
            "name": f"ENNProb_k{k}",
            "filter": ENNProbFilter(n_neighbors=k, n_jobs=-1),
        }) 
        for tau in thresholds:
            distance_grid.append({
                "name": f"ENNTh_k{k}_tau{tau}",
                "filter": ENNProbFilter(n_neighbors=k, mode="th", threshold=tau, n_jobs=-1),
            })

    #NCNEdit
    for k in ks:
        distance_grid.append({
            "name": f"NCNEdit_k{k}",
            "filter": NCNEdit(n_neighbors=k, n_jobs=-1),
        })

    # Multiedit 
    for k in ks:
        for b in n_blocks:
            for mi in max_iters:
                distance_grid.append({
                    "name": f"MultiEdit_k{k}_b{b}_mi{mi}",
                    "filter": MultiEditFilter(n_neighbors=k, n_blocks=b, n_jobs=-1, max_iter=mi),
                })

    return distance_grid

# Create the distance_based_filter_list
distance_grid = build_distance_grid(ks, thresholds, n_blocks)

# Specify dataset info
noise_type = "cla_rand"
folds = (1, 2, 3, 4, 5)

all_results = []

classifier = LogisticRegression()
classifier_name ="logReg"


for noise_k in [5, 25, 50]:
    for seed in [1]:
        all_results = []

        # Prepare the path where to store the data
        out_path = Path(f"./results/proposalEvaluation/{noise_type}/{noise_k}/seed0{seed}/{classifier_name}")
        out_path.mkdir(parents=True, exist_ok=True)

        # Test filters in each dataset
        for ds_idx, dataset in enumerate(keel_datasets):
            print(ds_idx,dataset)
            # Compute baseline (no filter) results
            baseline_df = run_5cv_baseline(
                dataset=dataset,
                noise_type=noise_type,
                seed=seed,
                classifier = classifier,
                k=noise_k,
                encoding="onehot",
                test_source="noisy",
                folds=folds,
            )

            # Compute results using filters
            distance_experiments_5 = [
                {
                    "dataset": dataset,
                    "noise_type": noise_type,
                    "seed": seed,
                    "k": noise_k,
                    "filters": {cfg["name"]: cfg["filter"]},
                    "encoding": "onehot",
                    "test_source": "clean",
                    "folds": folds,
                    "summarize": True,
                    "classifier": classifier,
                    "experiment_name": f"{dataset}|{noise_type}|{noise_k}|{cfg['name']}",
                }
                for cfg in distance_grid
            ]
            classification_df, removal_df, class_summary_df, removal_summary_df = run_5cv_grid(
                distance_experiments_5
            )
            all_results.append(
                {
                    "dataset": dataset,
                    "baseline_df": baseline_df,
                    "classification_df": classification_df,
                    "removal_df": removal_df,
                    "class_summary_df": class_summary_df,
                    "removal_summary_df": removal_summary_df,
                }
            )

            with open(out_path / "dists.pkl", "wb") as f:
                pickle.dump(all_results, f)

### Filtros basados en clasificadores

In [ ]:
# Rejilla global de hiperparámetros
cvs = [5, 7, 10]
thresholds = [0.5, 0.7]
max_iters = [3]
base_estimators = [
    ("DT", DecisionTreeClassifier(criterion="entropy", splitter="best", random_state=33)),
    ("c45", c45_like),
    ("KNN1", KNeighborsClassifier(n_neighbors=3)),
    ("LR", LogisticRegression(max_iter=1000)),
]

def build_classifier_grid(cvs, thresholds):
    ''' 
    Return classifier_based_filter_list
    '''
    classifier_grid = []
    
    # CF
    for cv in cvs: 
        for est_name, est in base_estimators:
            classifier_grid.append({
                "name": f"CF_{est_name}_cv{cv}",
                "filter": ClassificationFilter(
                    estimator=est,
                    cv=cv,
                    action="remove",
                    random_state=33,
                ),
            })
    
    #CVCF y CVCFTh 
    for cv in cvs: 
        classifier_grid.append({
            "name": f"CVCF_cv{cv}",
            "filter": CVCFFilter(
                estimator=c45_like,
                cv=cv,
                vote_rule="consensus",
                action="remove",
                random_state=33,
            ),
        })
        for tau in thresholds:
            classifier_grid.append({
                "name": f"CVCFTh_cv{cv}_tau{tau}",
                "filter": CVCFFilter(
                    estimator=c45_like,
                    cv=cv,
                    vote_rule="threshold",
                    threshold=tau,
                    action="remove",
                    random_state=33,
                ),
            })

    #EF y EFTh
    for cv in cvs: 
        classifier_grid.append({
            "name": f"Ensemble_cv{cv}",
            "filter": EnsembleFiltering(
                estimators=[est for _, est in base_estimators],
                cv=cv,
                mode="consensus",
                action="remove",
                random_state=33,
            ),
        })
        for tau in thresholds:
            classifier_grid.append({
                "name": f"EnsembleTh_cv{cv}_tau{tau}",
                "filter": EnsembleFiltering(
                    estimators=[est for _, est in base_estimators],
                    cv=cv,
                    mode="threshold",
                    threshold=tau,
                    action="remove",
                    random_state=33,
                ),
            })

    #IPF
    for cv in cvs: 
        for max_iter in max_iters:
            classifier_grid.append({
                "name": f"IPF_p{cv}_consensus",
                "filter": IterativePartitioningFilter(
                    estimator=c45_like,
                    n_partitions=cv,
                    vote_rule="consensus",
                    action="remove",
                    p_stop=0.03,
                    k_patience=3,
                    max_iter=max_iter,
                    random_state=33,
                ),
            })
            classifier_grid.append({
                "name": f"IPF_p{cv}_majority",
                "filter": IterativePartitioningFilter(
                    estimator=c45_like,
                    n_partitions=cv,
                    vote_rule="majority",
                    action="remove",
                    p_stop=0.03,
                    k_patience=3,
                    max_iter=max_iter,
                    random_state=33,
                ),
            })


    #INFFC e INFFCTh
    for cv in cvs: 
        for max_iter in max_iters:
            classifier_grid.append({
                "name": f"INFFC_cv{cv}",
                "filter": INFFCFilter(
                    estimators=[est for _, est in base_estimators],
                    cv=cv,
                    decision_rule="consensus",
                    action="remove",
                    max_iter=max_iter,
                    max_removed_frac=0.5,
                    random_state=33,
                ),
            })
            for tau in thresholds:
                classifier_grid.append({
                    "name": f"INFFCTh_cv{cv}_tau{tau}",
                    "filter": INFFCFilter(
                        estimators=[est for _, est in base_estimators],
                        cv=cv,
                        decision_rule="threshold",
                        threshold=tau,
                        action="remove",
                        max_iter=max_iter,
                        max_removed_frac=0.5,
                        random_state=33,
                    ),
                })
    return classifier_grid
    
classifier_grid = build_classifier_grid(cvs, thresholds)

# Specify dataset info
noise_type = "cla_rand"
folds = (1, 2, 3, 4, 5)

all_results = []

classifier = LogisticRegression()
classifier_name ="logReg"


for noise_k in [5, 25, 50]:
    for seed in [1]:
        all_results = []

        # Prepare the path where to store the data
        out_path = Path(f"./results/proposalEvaluation/{noise_type}/{noise_k}/seed0{seed}/{classifier_name}")
        out_path.mkdir(parents=True, exist_ok=True)
                
        for ds_idx,dataset in enumerate(keel_datasets):
            print(ds_idx, dataset)
            # baseline_df = run_5cv_baseline(
            #     dataset=dataset,
            #     noise_type=noise_type,
            #     seed=seed,
            #     k=noise_k,
            #     encoding="onehot",
            #     test_source="noisy",
            #     folds=folds,
            #     classifier = classifier
            # )
            classifier_experiments_5 = [
                {
                    "dataset": dataset,
                    "noise_type": noise_type,
                    "seed": seed,
                    "k": noise_k,
                    "filters": {cfg["name"]: cfg["filter"]},
                    "encoding": "onehot",
                    "test_source": "clean",
                    "folds": folds,
                    "summarize": True,
                    "classifier": classifier,
                    "experiment_name": f"{dataset}|{noise_type}|{noise_k}|{cfg['name']}",
                }
                for cfg in classifier_grid
            ]
            classification_df, removal_df, class_summary_df, removal_summary_df = run_5cv_grid(
                classifier_experiments_5,
                save_path="results_5cv",
                save_format="pickle",
                warnings_path="results_5cv/warnings_run_5cv_grid_classif.txt",
                clear_warnings_file=False,
            )
            all_results.append({
                "dataset": dataset,
                "baseline_df": {},
                "classification_df": classification_df,
                "removal_df": removal_df,
                "class_summary_df": class_summary_df,
                "removal_summary_df": removal_summary_df,
            })

            with open(out_path / "classif.pkl", "wb") as f:
                pickle.dump(all_results, f)

### Métodos de limpieza de ruido

In [ ]:
noise_type = "cla_rand"
seed = 1
noise_k = 5
folds = (1, 2, 3, 4, 5)

ks = [3,5,9]
cvs = [5,7,9]
max_iters = [3]

# Rejilla de CNCNOS
wiper_grid = [
    {
        "name": "CNCNOS_default",
        "filter": CNCNOS(
            base_neighbors=k,
            score_neighbors=k,
            cv=cv,
            max_iter=max_iter,
            stagnation_patience=2,
            min_class_count=2,
            random_state=33,
            n_jobs=-1,
        )   
    }
    for max_iter in max_iters
    for cv in cvs
    for k in ks

]

# Specify dataset info
noise_type = "cla_rand"
folds = (1, 2, 3, 4, 5)


classifier = LogisticRegression()
classifier_name ="logReg"


for noise_k in [5, 25, 50]:
    for seed in [1]:
        all_results = []

        # Prepare the path where to store the data
        out_path = Path(f"./results/proposalEvaluation/{noise_type}/{noise_k}/seed0{seed}/{classifier_name}")
        out_path.mkdir(parents=True, exist_ok=True)
        
        for ds_idx, dataset in enumerate(keel_datasets):
            print(ds_idx, dataset)

            # baseline_df = run_5cv_baseline(
            #     dataset=dataset,
            #     noise_type=noise_type,
            #     seed=seed,
            #     k=noise_k,
            #     encoding="onehot",
            #     test_source="noisy",
            #     folds=folds,
            #     classifier=classifier,
            # )

            cnc_experiments = [
                {
                    "dataset": dataset,
                    "noise_type": noise_type,
                    "seed": seed,
                    "k": noise_k,
                    "filters": {cfg["name"]: cfg["filter"]},
                    "encoding": "onehot",
                    "test_source": "clean",
                    "folds": folds,
                    "summarize": True,
                    "classifier": classifier,
                    "experiment_name": f"{dataset}|{noise_type}|{noise_k}|{cfg['name']}",
                }
                for cfg in wiper_grid
            ]

            classification_df, removal_df, class_summary_df, removal_summary_df = run_5cv_grid(
                cnc_experiments,
                save_path="results_5cv_cncnos",
                save_format="pickle",
                warnings_path="results_5cv_cncnos/warnings_run_5cv_grid_cncnos.txt",
                clear_warnings_file=False,
            )

            all_results.append({
                "dataset": dataset,
                "baseline_df": {},
                "classification_df": classification_df,
                "removal_df": removal_df,
                "class_summary_df": class_summary_df,
                "removal_summary_df": removal_summary_df,
            })

            with open(out_path / "cncnos.pkl", "wb") as f:
                pickle.dump(all_results, f)

# TABPFN (individual)

In [ ]:
from tabpfn import TabPFNClassifier

# Set noise_type to NCAR
noise_type = "cla_rand"
# Set the classifier (to use after filtering)
classifier = LogisticRegression()
classifier_name ="logReg"
cvs = [5,7,10]
thresholds = [0.5, 0.7]

def build_tabpfn_classifier_grid(cvs, thresholds):
    grid = []

    # ClassificationFilter con TabPFN
    for cv in cvs:
        grid.append({
            "name": f"TABPFN_CF_cv{cv}",
            "filter": ClassificationFilter(
                estimator=TabPFNClassifier(device="auto", random_state=33),
                cv=cv,
                action="remove",
                random_state=33,
            ),
        })

    # CVCF con TabPFN
    for cv in cvs:
        grid.append({
            "name": f"TABPFN_CVCF_cv{cv}",
            "filter": CVCFFilter(
                estimator=TabPFNClassifier(device="auto", random_state=33),
                cv=cv,
                vote_rule="consensus",
                action="remove",
                random_state=33,
            ),
        })
        for tau in thresholds:
            grid.append({
                "name": f"TABPFN_CVCFTh_cv{cv}_tau{tau}",
                "filter": CVCFFilter(
                    estimator=TabPFNClassifier(device="auto", random_state=33),
                    cv=cv,
                    vote_rule="threshold",
                    threshold=tau,
                    action="remove",
                    random_state=33,
                ),
            })

    # IPF con TabPFN
    for cv in cvs:
        grid.append({
            "name": f"TABPFN_IPF_cv{cv}_consensus",
            "filter": IterativePartitioningFilter(
                estimator=TabPFNClassifier(device="auto", random_state=33),
                n_partitions=cv,
                vote_rule="consensus",
                action="remove",
                p_stop=0.03,
                k_patience=3,
                max_iter=3,
                random_state=33,
            ),
        })
        grid.append({
            "name": f"TABPFN_IPF_cv{cv}_majority",
            "filter": IterativePartitioningFilter(
                estimator=TabPFNClassifier(device="auto", random_state=33),
                n_partitions=cv,
                vote_rule="majority",
                action="remove",
                p_stop=0.03,
                k_patience=3,
                max_iter=3,
                random_state=33,
            ),
        })

    return grid


tabpfn_classifier_grid = build_tabpfn_classifier_grid(cvs, thresholds)


for noise_k in [5, 25, 50]:
    for seed in [1]:
        all_results = []

        # Prepare the path where to store the data
        out_path = Path(f"./results/proposalEvaluation/{noise_type}/{noise_k}/seed0{seed}/{classifier_name}")
        out_path.mkdir(parents=True, exist_ok=True)

        for ds_idx, dataset in enumerate(keel_datasets):
            print(ds_idx, dataset)
            # baseline_df = run_5cv_baseline(
            #     dataset=dataset,
            #     noise_type=noise_type,
            #     seed=seed,
            #     classifier=classifier,
            #     k=noise_k,
            #     encoding="onehot",
            #     test_source="noisy",
            #     folds=(1, 2, 3, 4, 5),
            # )

            tabpfn_experiments_5 = [
                {
                    "dataset": dataset,
                    "noise_type": noise_type,
                    "seed": seed,
                    "k": noise_k,
                    "filters": {cfg["name"]: cfg["filter"]},
                    "encoding": "onehot",
                    "test_source": "clean",
                    "folds": (1, 2, 3, 4, 5),
                    "summarize": True,
                    "classifier": classifier,
                    "experiment_name": f"{dataset}|{noise_type}|{noise_k}|{cfg['name']}",
                }
                for cfg in tabpfn_classifier_grid
            ]

            classification_df, removal_df, class_summary_df, removal_summary_df = run_5cv_grid(
                tabpfn_experiments_5
            )

            all_results.append({
                "dataset": dataset,
                "baseline_df": {},
                "classification_df": classification_df,
                "removal_df": removal_df,
                "class_summary_df": class_summary_df,
                "removal_summary_df": removal_summary_df,
            })

            with open(out_path / "TabPFN.pkl", "wb") as f:
                pickle.dump(all_results, f)